# 자연어 명령으로 핑키 주행 제어하기

- 목적: OpenAI Tool Calling으로 자연어 주행 명령을 해석하고, ROS2 Nav2 또는 `/cmd_vel` 제어 함수로 연결합니다.
- 사용 방식:
  - 목적지 이동: `Nav2 BasicNavigator` 사용
  - 여러 좌표 이동: `followWaypoints()` 사용
  - 짧은 수동 이동: `/cmd_vel` 직접 publish 사용
- 예시 명령:
  - `point1 위치로 이동해.`
  - `x=0.5, y=0 위치로 이동해. 방향은 0도로 해.`
  - `point1, point2, point3 순서로 이동해.`
  - `앞으로 조금 가.`
  - `왼쪽으로 천천히 돌아.`
  - `멈춰.`

```text
자연어 명령
    ↓
OpenAI Responses API
    ↓
function_call 생성
    ↓
Python 함수 실행
    ↓
Nav2 goal 전송 또는 /cmd_vel publish
```

## 0. 실행 전 준비

- 로봇에서 bringup을 실행합니다.

```bash
ros2 launch pinky_bringup bringup_robot.launch.xml
```

- 로봇에서 navigation을 실행합니다.

```bash
ros2 launch pinky_navigation bringup_launch.xml map:=<저장한_맵이름.yaml>
```

- PC에서 RViz를 실행합니다.

```bash
ros2 launch pinky_navigation nav2_view.launch.xml
```

- RViz에서 다음 작업을 수행합니다.
  - `2D Pose Estimate`로 라이다와 맵을 맞춥니다.
  - `Publish Point`로 이동할 위치를 찍습니다.
  - 필요한 경우 `/clicked_point`로 좌표를 확인합니다.

```bash
ros2 topic echo /clicked_point
```

- `/clicked_point`에서 나온 `z` 좌표는 사용하지 않습니다.
- 주행 목표에는 `x`, `y` 좌표를 사용합니다.

## 1. OpenAI API 설정

- `API_KEY`에 본인의 OpenAI API Key를 입력합니다.
- 예제 파일에는 실제 API Key를 저장하지 않습니다.

In [ ]:
from openai import OpenAI
import json

API_KEY = "여기에_API_KEY_입력"

if API_KEY == "여기에_API_KEY_입력":
    raise ValueError("API_KEY 값을 본인의 OpenAI API Key로 바꾼 뒤 실행합니다.")

client = OpenAI(
    api_key=API_KEY
)

MODEL = "gpt-5.5"

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

client = OpenAI()

MODEL = "gpt-5.5"

## 2. OpenAI API 연결 테스트

- API Key가 정상인지 확인합니다.
- 짧은 문장이 출력되면 다음 단계로 진행합니다.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="안녕. OpenAI API 연결 테스트야. 한 문장으로 짧게 답해줘."
)

print(response.output_text)

## 3. ROS2/Nav2 주행 라이브러리 불러오기

- 목적지 주행에는 `BasicNavigator`와 `PoseStamped`를 사용합니다.
- 짧은 수동 이동에는 `/cmd_vel`과 `Twist`를 사용합니다.
- 시간 제한과 결과 확인에는 `Duration`, `TaskResult`를 사용합니다.

In [ ]:
import math
import time

import rclpy
from rclpy.duration import Duration

from geometry_msgs.msg import PoseStamped, Twist
from nav2_simple_commander.robot_navigator import BasicNavigator, TaskResult

import tf_transformations

## 4. Nav2 실행 확인

- `BasicNavigator()` 객체를 생성합니다.
- `waitUntilNav2Active()`로 Nav2가 준비될 때까지 기다립니다.
- 이 셀은 `pinky_navigation bringup_launch.xml` 실행 후 사용합니다.

In [ ]:
if not rclpy.ok():
    rclpy.init()

nav = BasicNavigator()
nav.waitUntilNav2Active()

print("Nav2 is ready for use.")

## 5. `/cmd_vel` publisher 준비

- `/cmd_vel`은 로봇 속도 명령 토픽입니다.
- `Twist.linear.x`는 전진/후진 속도입니다.
- `Twist.angular.z`는 제자리 회전 속도입니다.
- 짧은 수동 명령에만 사용합니다.

In [ ]:
cmd_vel_pub = nav.create_publisher(Twist, "cmd_vel", 10)

print("/cmd_vel publisher is ready.")

## 6. degree 각도를 quaternion으로 변환하는 함수

- 사람이 이해하기 쉬운 각도는 degree입니다.
- Nav2 goal에는 quaternion 형식의 방향값이 필요합니다.
- `yaw_deg`를 quaternion으로 변환합니다.

In [ ]:
def get_quaternion_from_yaw(yaw_degrees: float):
    yaw_radians = math.radians(yaw_degrees)
    quaternion = tf_transformations.quaternion_from_euler(0, 0, yaw_radians)
    return quaternion

## 7. Nav2 goal pose 생성 함수

- `frame_id`는 `map`으로 설정합니다.
- `x`, `y`는 목표 위치입니다.
- `yaw_deg`는 목표 방향입니다.

In [ ]:
def make_goal_pose(x: float, y: float, yaw_deg: float = 0.0) -> PoseStamped:
    q = get_quaternion_from_yaw(yaw_deg)

    goal_pose = PoseStamped()
    goal_pose.header.frame_id = "map"
    goal_pose.header.stamp = nav.get_clock().now().to_msg()

    goal_pose.pose.position.x = float(x)
    goal_pose.pose.position.y = float(y)
    goal_pose.pose.position.z = 0.0

    goal_pose.pose.orientation.x = q[0]
    goal_pose.pose.orientation.y = q[1]
    goal_pose.pose.orientation.z = q[2]
    goal_pose.pose.orientation.w = q[3]

    return goal_pose

## 8. 주행 결과 변환 함수

- `nav.getResult()` 결과를 문자열로 변환합니다.
- 출력 결과를 읽기 쉽게 정리합니다.

In [ ]:
def task_result_to_text(result) -> str:
    if result == TaskResult.SUCCEEDED:
        return "SUCCEEDED"

    if result == TaskResult.CANCELED:
        return "CANCELED"

    if result == TaskResult.FAILED:
        return "FAILED"

    return f"UNKNOWN: {result}"

## 9. 단일 목표 지점으로 이동하는 함수

- `nav.goToPose(goal_pose)`로 한 지점까지 이동합니다.
- 주행 중 남은 거리를 출력합니다.
- `timeout_sec`을 넘으면 주행을 취소합니다.

In [ ]:
def navigate_to_pose(x: float, y: float, yaw_deg: float = 0.0, timeout_sec: int = 300):
    goal_pose = make_goal_pose(x, y, yaw_deg)

    print(f"[Nav2] goToPose: x={x}, y={y}, yaw_deg={yaw_deg}")
    nav.goToPose(goal_pose)

    while not nav.isTaskComplete():
        rclpy.spin_once(nav, timeout_sec=1.0)

        feedback = nav.getFeedback()

        if feedback is not None:
            try:
                print(
                    "Distance remaining:",
                    round(feedback.distance_remaining, 3),
                    "m"
                )
            except Exception:
                pass

            if Duration.from_msg(feedback.navigation_time) > Duration(seconds=timeout_sec):
                nav.cancelTask()
                publish_zero_velocity()
                return {
                    "success": False,
                    "action": "navigate_to_pose",
                    "result": "CANCELED",
                    "message": f"{timeout_sec}초를 초과해서 주행을 취소했습니다.",
                    "target": {
                        "x": x,
                        "y": y,
                        "yaw_deg": yaw_deg
                    }
                }

    result = nav.getResult()
    result_text = task_result_to_text(result)

    return {
        "success": result == TaskResult.SUCCEEDED,
        "action": "navigate_to_pose",
        "result": result_text,
        "target": {
            "x": x,
            "y": y,
            "yaw_deg": yaw_deg
        }
    }

## 10. 예시 좌표를 등록 위치로 만들기

- RViz의 `Publish Point`로 좌표를 찍습니다.
- `/clicked_point`에서 확인한 좌표를 아래 값에 반영합니다.
- 실제 실습 환경에 맞게 `point1`, `point2`, `point3` 값을 수정합니다.

In [ ]:
NAMED_LOCATIONS = {
    "point1": {"x": 1.1, "y": 1.0, "yaw_deg": 0},
    "point2": {"x": -1.4, "y": -1.0, "yaw_deg": 180},
    "point3": {"x": -1.35, "y": -0.30, "yaw_deg": 45},
}


def navigate_to_named_location(location_name: str, timeout_sec: int = 300):
    if location_name not in NAMED_LOCATIONS:
        return {
            "success": False,
            "action": "navigate_to_named_location",
            "message": f"등록되지 않은 위치입니다: {location_name}",
            "available_locations": list(NAMED_LOCATIONS.keys())
        }

    pose = NAMED_LOCATIONS[location_name]

    result = navigate_to_pose(
        x=pose["x"],
        y=pose["y"],
        yaw_deg=pose["yaw_deg"],
        timeout_sec=timeout_sec
    )

    result["location_name"] = location_name
    return result

## 11. 여러 좌표를 순서대로 주행하는 함수

- `goal_pose_list`를 만듭니다.
- `nav.followWaypoints(goal_pose_list)`로 순서대로 주행합니다.
- 자연어 예시: `point1, point2, point3 순서로 이동해.`
- `FollowWaypoints` feedback에는 `navigation_time`이 없으므로, timeout은 시작 시각 기준 경과 시간으로 계산합니다.

In [ ]:
def follow_waypoints(points: list, timeout_sec: int = 300):
    goal_pose_list = []

    for point in points:
        x = point["x"]
        y = point["y"]
        yaw_deg = point.get("yaw_deg", 0.0)

        goal_pose = make_goal_pose(x, y, yaw_deg)
        goal_pose_list.append(goal_pose)

    print(f"[Nav2] followWaypoints: {len(goal_pose_list)} points")

    # FollowWaypoints의 feedback에는 NavigateToPose와 달리 navigation_time이 없습니다.
    # 따라서 timeout은 feedback.navigation_time이 아니라 시작 시각 기준 경과 시간으로 계산합니다.
    start_time = nav.get_clock().now()

    nav.followWaypoints(goal_pose_list)

    while not nav.isTaskComplete():
        rclpy.spin_once(nav, timeout_sec=1.0)

        feedback = nav.getFeedback()

        if feedback is not None:
            try:
                print("Current waypoint:", feedback.current_waypoint + 1)
            except Exception:
                pass

        elapsed_sec = (nav.get_clock().now() - start_time).nanoseconds / 1_000_000_000

        if elapsed_sec > timeout_sec:
            nav.cancelTask()
            publish_zero_velocity()
            return {
                "success": False,
                "action": "follow_waypoints",
                "result": "CANCELED",
                "message": f"{timeout_sec}초를 초과해서 waypoint 주행을 취소했습니다.",
                "points": points
            }

    result = nav.getResult()
    result_text = task_result_to_text(result)

    return {
        "success": result == TaskResult.SUCCEEDED,
        "action": "follow_waypoints",
        "result": result_text,
        "points": points
    }

## 12. `/cmd_vel` 직접 제어 함수

- 짧은 수동 이동에 사용합니다.
- 예시 명령:
  - `앞으로 조금 가.`
  - `뒤로 조금 가.`
  - `왼쪽으로 천천히 돌아.`
  - `오른쪽으로 천천히 돌아.`
  - `멈춰.`
- 안전을 위해 속도와 시간을 제한합니다.
- `/cmd_vel` 명령을 실행하기 전에 Nav2 작업을 취소합니다.

In [ ]:
MAX_LINEAR_X = 0.12
MAX_ANGULAR_Z = 0.50
MAX_DIRECT_DRIVE_DURATION_SEC = 3.0
CMD_VEL_RATE_HZ = 10.0

DIRECT_DRIVE_SPEEDS = {
    "slow": {
        "linear_x": 0.05,
        "angular_z": 0.25
    },
    "normal": {
        "linear_x": 0.10,
        "angular_z": 0.45
    }
}


def clamp(value: float, min_value: float, max_value: float) -> float:
    return max(min(float(value), max_value), min_value)


def publish_cmd_vel(linear_x: float = 0.0, angular_z: float = 0.0):
    msg = Twist()

    msg.linear.x = clamp(linear_x, -MAX_LINEAR_X, MAX_LINEAR_X)
    msg.angular.z = clamp(angular_z, -MAX_ANGULAR_Z, MAX_ANGULAR_Z)

    cmd_vel_pub.publish(msg)


def publish_zero_velocity():
    publish_cmd_vel(0.0, 0.0)


def direct_drive_motion(motion: str, speed: str, duration_sec: float):
    if motion not in ["forward", "backward", "turn_left", "turn_right"]:
        return {
            "success": False,
            "action": "direct_drive_motion",
            "message": f"지원하지 않는 동작입니다: {motion}"
        }

    if speed not in DIRECT_DRIVE_SPEEDS:
        return {
            "success": False,
            "action": "direct_drive_motion",
            "message": f"지원하지 않는 속도입니다: {speed}"
        }

    duration_sec = clamp(duration_sec, 0.1, MAX_DIRECT_DRIVE_DURATION_SEC)

    linear_x = 0.0
    angular_z = 0.0

    if motion == "forward":
        linear_x = DIRECT_DRIVE_SPEEDS[speed]["linear_x"]

    elif motion == "backward":
        linear_x = -DIRECT_DRIVE_SPEEDS[speed]["linear_x"]

    elif motion == "turn_left":
        angular_z = DIRECT_DRIVE_SPEEDS[speed]["angular_z"]

    elif motion == "turn_right":
        angular_z = -DIRECT_DRIVE_SPEEDS[speed]["angular_z"]

    # Nav2 goal과 /cmd_vel 직접 제어가 동시에 동작하지 않도록 취소합니다.
    try:
        nav.cancelTask()
    except Exception:
        pass

    steps = max(1, int(duration_sec * CMD_VEL_RATE_HZ))
    sleep_time = 1.0 / CMD_VEL_RATE_HZ

    print(
        f"[cmd_vel] motion={motion}, speed={speed}, "
        f"linear_x={linear_x}, angular_z={angular_z}, duration={duration_sec}"
    )

    for _ in range(steps):
        publish_cmd_vel(linear_x=linear_x, angular_z=angular_z)
        rclpy.spin_once(nav, timeout_sec=0.01)
        time.sleep(sleep_time)

    publish_zero_velocity()

    return {
        "success": True,
        "action": "direct_drive_motion",
        "motion": motion,
        "speed": speed,
        "duration_sec": duration_sec,
        "linear_x": linear_x,
        "angular_z": angular_z,
        "message": "cmd_vel 직접 제어 명령을 실행했습니다."
    }

## 13. 정지 함수

- `멈춰`, `정지`, `취소`, `stop` 명령에 사용합니다.
- Nav2 작업을 취소합니다.
- `/cmd_vel`을 0으로 publish합니다.

In [ ]:
def stop_navigation():
    try:
        nav.cancelTask()
    except Exception:
        pass

    publish_zero_velocity()

    return {
        "success": True,
        "action": "stop_navigation",
        "message": "Nav2 주행 취소 및 cmd_vel 정지 명령을 보냈습니다."
    }

## 14. 상태 확인 함수

- 등록된 위치 목록을 반환합니다.
- 주행 명령에 필요한 최소 상태만 제공합니다.

In [ ]:
def get_navigation_status():
    return {
        "success": True,
        "action": "get_navigation_status",
        "available_locations": list(NAMED_LOCATIONS.keys()),
        "direct_drive_commands": [
            "forward",
            "backward",
            "turn_left",
            "turn_right"
        ]
    }

## 15. OpenAI Tool 정의

- 모델에게 사용할 수 있는 함수 목록을 알려줍니다.
- 모델은 함수를 직접 실행하지 않습니다.
- 실제 실행은 Python 코드가 수행합니다.

In [ ]:
tools = [
    {
        "type": "function",
        "name": "navigate_to_named_location",
        "description": "등록된 위치 이름으로 로봇을 이동시킨다. 사용 가능한 위치: point1, point2, point3",
        "parameters": {
            "type": "object",
            "properties": {
                "location_name": {
                    "type": "string",
                    "enum": list(NAMED_LOCATIONS.keys()),
                    "description": "이동할 등록 위치 이름"
                }
            },
            "required": ["location_name"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "navigate_to_pose",
        "description": "로봇을 map 좌표계 기준의 특정 x, y, yaw_deg 위치로 이동시킨다.",
        "parameters": {
            "type": "object",
            "properties": {
                "x": {
                    "type": "number",
                    "description": "map 좌표계 기준 목표 x 좌표"
                },
                "y": {
                    "type": "number",
                    "description": "map 좌표계 기준 목표 y 좌표"
                },
                "yaw_deg": {
                    "type": "number",
                    "description": "목표 방향 각도. 단위는 degree"
                }
            },
            "required": ["x", "y", "yaw_deg"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "follow_waypoints",
        "description": "여러 좌표를 순서대로 주행한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "points": {
                    "type": "array",
                    "description": "순서대로 이동할 좌표 목록",
                    "items": {
                        "type": "object",
                        "properties": {
                            "x": {"type": "number"},
                            "y": {"type": "number"},
                            "yaw_deg": {"type": "number"}
                        },
                        "required": ["x", "y", "yaw_deg"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["points"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "direct_drive_motion",
        "description": "짧은 수동 이동을 위해 /cmd_vel을 직접 publish한다. 예: 앞으로 조금 가기, 뒤로 조금 가기, 왼쪽으로 천천히 돌기, 오른쪽으로 천천히 돌기",
        "parameters": {
            "type": "object",
            "properties": {
                "motion": {
                    "type": "string",
                    "enum": ["forward", "backward", "turn_left", "turn_right"],
                    "description": "수동 이동 방향"
                },
                "speed": {
                    "type": "string",
                    "enum": ["slow", "normal"],
                    "description": "수동 이동 속도"
                },
                "duration_sec": {
                    "type": "number",
                    "description": "동작 시간. 단위는 초. 0.1초 이상 3.0초 이하로 사용한다."
                }
            },
            "required": ["motion", "speed", "duration_sec"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "stop_navigation",
        "description": "현재 Nav2 주행을 취소하고 /cmd_vel을 0으로 만들어 로봇을 정지한다.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "get_navigation_status",
        "description": "등록된 위치 목록과 직접 제어 명령 목록을 확인한다.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        },
        "strict": True
    }
]

## 16. function_call 실행 함수

- 모델이 선택한 함수 이름을 확인합니다.
- 실제 Python 함수를 실행합니다.
- 실행 결과를 다시 모델에 전달할 수 있는 형태로 반환합니다.

In [ ]:
def run_navigation_function(function_name: str, arguments: dict):
    if function_name == "navigate_to_named_location":
        return navigate_to_named_location(
            location_name=arguments["location_name"]
        )

    if function_name == "navigate_to_pose":
        return navigate_to_pose(
            x=arguments["x"],
            y=arguments["y"],
            yaw_deg=arguments["yaw_deg"]
        )

    if function_name == "follow_waypoints":
        return follow_waypoints(
            points=arguments["points"]
        )

    if function_name == "direct_drive_motion":
        return direct_drive_motion(
            motion=arguments["motion"],
            speed=arguments["speed"],
            duration_sec=arguments["duration_sec"]
        )

    if function_name == "stop_navigation":
        return stop_navigation()

    if function_name == "get_navigation_status":
        return get_navigation_status()

    return {
        "success": False,
        "message": f"알 수 없는 함수입니다: {function_name}"
    }

## 17. 자연어 주행 명령 실행 함수

- 사용자 명령을 OpenAI 모델에 전달합니다.
- 모델이 적절한 tool을 선택합니다.
- Python 코드가 선택된 함수를 실행합니다.
- 함수 실행 결과를 바탕으로 최종 답변을 생성합니다.

In [ ]:
def ask_navigation(command: str):
    instructions = f'''
너는 ROS2 Nav2 기반 핑키 자율주행 명령 해석기다.

규칙:
1. 등록된 위치로 이동하라는 명령에는 navigate_to_named_location을 사용한다.
2. 등록된 위치 이름은 {list(NAMED_LOCATIONS.keys())} 이다.
3. x, y, yaw_deg가 명확한 명령에는 navigate_to_pose를 사용한다.
4. 여러 좌표를 순서대로 이동하라는 명령에는 follow_waypoints를 사용한다.
5. "앞으로 조금 가", "뒤로 조금 가", "왼쪽으로 천천히 돌아", "오른쪽으로 천천히 돌아" 같은 짧은 수동 이동에는 direct_drive_motion을 사용한다.
6. "조금"은 duration_sec 0.8~1.2 사이로 해석한다.
7. "천천히"는 speed="slow"로 해석한다.
8. 속도 표현이 없으면 speed="slow"를 우선 사용한다.
9. direct_drive_motion의 duration_sec는 3.0을 넘기지 않는다.
10. "멈춰", "정지", "취소", "stop"은 stop_navigation을 사용한다.
11. 가능한 위치나 상태를 묻는 명령에는 get_navigation_status를 사용한다.
12. 좌표, 위치, 이동 방향이 애매하면 함수를 호출하지 말고 확인 질문을 한다.
13. LCD, LED, 감정 행동은 사용하지 않는다.
'''

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=command,
        tools=tools,
        tool_choice="auto",
        parallel_tool_calls=False,
        max_output_tokens=600,
    )

    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        return response.output_text

    function_outputs = []

    for call in function_calls:
        args = json.loads(call.arguments)

        print(f"[모델이 선택한 함수] {call.name}")
        print(f"[함수 인자] {args}")

        result = run_navigation_function(call.name, args)

        print(f"[함수 실행 결과] {result}")

        function_outputs.append(
            {
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, ensure_ascii=False)
            }
        )

    final_response = client.responses.create(
        model=MODEL,
        previous_response_id=response.id,
        input=function_outputs,
        tools=tools,
        max_output_tokens=600,
    )

    return final_response.output_text

## 18. 실행 예시 1: 등록 위치로 이동

- `point1`, `point2`, `point3` 중 하나로 이동합니다.

In [ ]:
answer = ask_navigation("point1 위치로 이동해.")
print("\n[최종 답변]")
print(answer)

## 19. 실행 예시 2: 좌표를 직접 지정해서 이동

- RViz의 `Publish Point` 또는 `/clicked_point`로 확인한 좌표를 사용합니다.

In [ ]:
answer = ask_navigation("x=0.5, y=0 위치로 이동해. 방향은 0도로 해.")
print("\n[최종 답변]")
print(answer)

## 20. 실행 예시 3: 여러 좌표 순서대로 주행

- 여러 goal을 순서대로 보냅니다.

In [ ]:
answer = ask_navigation("point1, point2, point3 순서로 이동해.")
print("\n[최종 답변]")
print(answer)

## 21. 실행 예시 4: 앞으로 조금 이동

- `/cmd_vel` 직접 제어를 사용합니다.
- 짧은 전진 동작입니다.

In [ ]:
answer = ask_navigation("앞으로 조금 가.")
print("\n[최종 답변]")
print(answer)

## 22. 실행 예시 5: 왼쪽으로 천천히 회전

- `/cmd_vel` 직접 제어를 사용합니다.
- 제자리 좌회전 동작입니다.

In [ ]:
answer = ask_navigation("왼쪽으로 천천히 돌아.")
print("\n[최종 답변]")
print(answer)

## 23. 실행 예시 6: 정지

- Nav2 goal을 취소합니다.
- `/cmd_vel`을 0으로 보냅니다.

In [ ]:
answer = ask_navigation("멈춰.")
print("\n[최종 답변]")
print(answer)

## 24. 실행 예시 7: 등록 위치와 직접 제어 명령 확인

- 현재 사용할 수 있는 등록 위치와 직접 제어 명령을 확인합니다.

In [ ]:
answer = ask_navigation("내가 사용할 수 있는 이동 명령을 알려줘.")
print("\n[최종 답변]")
print(answer)

## 25. Nav2 종료

- 실습 종료 시 실행합니다.
- 다른 노트북이나 노드에서 Nav2를 계속 사용할 경우에는 실행하지 않습니다.

In [ ]:
# 실습 종료 시에만 실행합니다.
nav.lifecycleShutdown()
publish_zero_velocity()
rclpy.shutdown()

## 26. YOLO11로 본 내용을 바탕으로 대화하기

- 목적: 카메라로 한 장의 이미지를 촬영하고, YOLO11로 물체를 검출한 뒤, 검출 결과를 바탕으로 OpenAI 모델과 대화합니다.
- 참고 구조:
  - `Camera()`로 카메라를 엽니다.
  - `cam.start()`로 카메라를 시작합니다.
  - `cam.get_frame()`으로 프레임 한 장을 가져옵니다.
  - `YOLO("yolo11n.pt")`로 모델을 불러옵니다.
  - `model.predict()`로 객체를 검출합니다.
  - 검출된 물체 이름, 신뢰도, 박스 좌표를 텍스트로 정리합니다.
  - 정리된 결과를 OpenAI 모델에 전달하여 대화합니다.

```text
카메라 프레임
    ↓
YOLO11 객체 검출
    ↓
검출 결과 요약
    ↓
OpenAI 대화
    ↓
"앞에 사람이 보입니다.", "책상 위에 컵이 있습니다." 같은 답변 생성
```

- 이 섹션은 기존 주행 명령 코드를 수정하지 않습니다.
- 이 섹션은 기존 `client`와 `MODEL` 변수를 그대로 사용합니다.

## 27. YOLO11 패키지와 카메라 라이브러리 불러오기

- `ultralytics`는 YOLO11 모델 실행에 사용합니다.
- `PIL.Image`와 `IPython.display`는 이미지를 노트북에 표시하는 데 사용합니다.
- `pinkylib.Camera`는 핑키 카메라 프레임을 가져오는 데 사용합니다.

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display, clear_output
import io
import os

from pinkylib import Camera

## 28. 카메라 프레임 한 장 가져오기

- `cam.get_frame()`으로 카메라 프레임을 가져옵니다.
- 카메라 프레임은 BGR 순서입니다.
- 화면에 표시할 때는 RGB 순서로 바꿉니다.

In [ ]:
def capture_one_frame(cam):
    frame_bgr = cam.get_frame()
    image_rgb = Image.fromarray(frame_bgr[:, :, ::-1])
    return frame_bgr, image_rgb


def test_camera_once():
    cam = Camera()
    cam.start()

    try:
        frame_bgr, image_rgb = capture_one_frame(cam)
        display(image_rgb)
        return frame_bgr

    finally:
        cam.close()

## 29. YOLO11n 모델 불러오기

- 처음 실행할 때 `yolo11n.pt` 파일이 자동으로 다운로드될 수 있습니다.
- 인터넷이 없는 환경에서는 `yolo11n.pt` 파일을 미리 준비해서 노트북과 같은 폴더에 둡니다.
- 라즈베리파이에서는 먼저 `yolo11n.pt`처럼 작은 모델로 실습합니다.

In [ ]:
yolo_model = YOLO("yolo11n.pt")
print("YOLO model loaded.")

## 30. 카메라 이미지 한 장에서 YOLO 검출하기

- 카메라에서 한 장을 촬영합니다.
- YOLO11로 객체를 검출합니다.
- 원본 이미지와 검출 박스가 그려진 이미지를 표시합니다.

In [ ]:
def detect_once_from_camera(imgsz=320, conf=0.25, show=True):
    cam = Camera()
    cam.start()

    try:
        frame_bgr, image_rgb = capture_one_frame(cam)

        results = yolo_model.predict(
            source=frame_bgr,
            imgsz=imgsz,
            conf=conf,
            verbose=False
        )

        result = results[0]
        annotated_bgr = result.plot()
        annotated_rgb = Image.fromarray(annotated_bgr[:, :, ::-1])

        if show:
            print("[원본 이미지]")
            display(image_rgb)
            print("[YOLO 검출 결과]")
            display(annotated_rgb)

        return result, annotated_bgr

    finally:
        cam.close()

## 31. YOLO 검출 결과를 텍스트로 정리하기

- OpenAI 모델은 YOLO 결과 객체를 직접 읽지 못합니다.
- 따라서 검출 결과를 사람이 읽을 수 있는 텍스트/딕셔너리 형태로 바꿉니다.
- 포함 정보:
  - 물체 이름
  - 신뢰도
  - bounding box 좌표
  - 화면 중심 기준 위치

In [ ]:
def summarize_yolo_result(result, max_objects=10):
    detections = []

    if result.boxes is None or len(result.boxes) == 0:
        return {
            "count": 0,
            "detections": [],
            "summary_text": "탐지된 물체가 없습니다."
        }

    # 이미지 크기 확인
    image_height, image_width = result.orig_shape[:2]
    center_x = image_width / 2

    for i, box in enumerate(result.boxes[:max_objects], start=1):
        cls_id = int(box.cls[0])
        name = yolo_model.names[cls_id]
        conf = float(box.conf[0])

        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
        box_center_x = (x1 + x2) / 2

        if box_center_x < center_x * 0.8:
            horizontal_position = "left"
        elif box_center_x > center_x * 1.2:
            horizontal_position = "right"
        else:
            horizontal_position = "center"

        detections.append(
            {
                "index": i,
                "name": name,
                "confidence": round(conf, 3),
                "box_xyxy": [
                    round(x1, 1),
                    round(y1, 1),
                    round(x2, 1),
                    round(y2, 1),
                ],
                "horizontal_position": horizontal_position
            }
        )

    lines = []

    for item in detections:
        lines.append(
            f'{item["index"]}. {item["name"]} '
            f'(confidence={item["confidence"]}, '
            f'position={item["horizontal_position"]}, '
            f'box={item["box_xyxy"]})'
        )

    return {
        "count": len(detections),
        "detections": detections,
        "summary_text": "\n".join(lines)
    }


def print_yolo_summary(summary):
    print(f"탐지 개수: {summary['count']}")

    if summary["count"] == 0:
        print(summary["summary_text"])
        return

    print(summary["summary_text"])

## 32. YOLO 검출 결과 확인

- 아래 셀은 카메라 한 장을 촬영하고 객체를 검출합니다.
- `last_yolo_summary`에 검출 결과가 저장됩니다.

In [ ]:
last_yolo_result, last_yolo_annotated_bgr = detect_once_from_camera(
    imgsz=320,
    conf=0.25,
    show=True
)

last_yolo_summary = summarize_yolo_result(last_yolo_result)
print_yolo_summary(last_yolo_summary)

## 33. YOLO 검출 결과를 바탕으로 OpenAI와 대화하기

- 사용자의 질문과 YOLO 검출 결과를 함께 OpenAI 모델에 전달합니다.
- 모델은 실제 이미지를 보는 것이 아니라, YOLO가 정리한 검출 결과 텍스트를 보고 답합니다.
- 예시 질문:
  - `앞에 뭐가 보여?`
  - `사람이 보이면 조심하라고 말해줘.`
  - `검출된 물체를 설명해줘.`

In [ ]:
def ask_about_yolo_detection(question: str, yolo_summary: dict = None):
    if yolo_summary is None:
        if "last_yolo_summary" not in globals():
            return "먼저 YOLO 검출 셀을 실행해야 합니다."

        yolo_summary = last_yolo_summary

    instructions = '''
너는 로봇 카메라의 YOLO 객체 검출 결과를 설명하는 보조자다.

규칙:
1. 답변은 한국어로 한다.
2. YOLO 검출 결과에 있는 물체만 근거로 답한다.
3. 검출 결과에 없는 물체를 보았다고 말하지 않는다.
4. confidence가 낮을 수 있으므로 단정하지 않고 "보입니다", "탐지되었습니다"처럼 표현한다.
5. 위치 정보는 left, center, right를 기준으로 설명한다.
6. 주행 명령은 직접 실행하지 않는다.
7. 위험 판단이 필요한 경우 사람, 의자, 컵, 병, 가방 등 장애물이 될 수 있는 물체를 언급한다.
'''

    detection_text = yolo_summary.get("summary_text", "탐지 결과가 없습니다.")

    prompt = f'''
사용자 질문:
{question}

YOLO 검출 결과:
{detection_text}
'''

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=prompt,
        max_output_tokens=500,
    )

    return response.output_text

## 34. 카메라를 새로 보고 바로 대화하기

- 카메라 촬영, YOLO 검출, OpenAI 대화를 한 번에 실행합니다.
- 매번 현재 카메라 화면을 새로 보고 답변합니다.

In [ ]:
def look_and_talk(question: str, imgsz=320, conf=0.25, show=True):
    result, annotated_bgr = detect_once_from_camera(
        imgsz=imgsz,
        conf=conf,
        show=show
    )

    summary = summarize_yolo_result(result)

    print("[YOLO 검출 요약]")
    print_yolo_summary(summary)

    answer = ask_about_yolo_detection(
        question=question,
        yolo_summary=summary
    )

    return answer, summary, annotated_bgr

## 35. 실행 예시: 현재 화면을 새로 보고 답하기

- 카메라로 새 프레임을 촬영합니다.
- YOLO11로 객체를 검출합니다.
- 검출 결과를 바탕으로 답변합니다.

In [ ]:
answer, current_yolo_summary, current_yolo_annotated_bgr = look_and_talk(
    "지금 로봇 앞에 무엇이 있는지 말해줘.",
    imgsz=320,
    conf=0.25,
    show=True
)

print("\n[OpenAI 답변]")
print(answer)

## 36. YOLO 결과와 주행 명령을 함께 사용

- YOLO 결과 기반 대화와 주행 명령을 분리합니다.
- YOLO 결과는 상황 설명에 사용합니다.
- 주행 실행은 기존 `ask_navigation()` 함수로 수행합니다.
- 안전을 위해 다음 원칙을 지킵니다.
  - 사람이 탐지되면 먼저 정지 또는 확인 질문을 사용합니다.
  - 장애물이 보이면 바로 전진 명령을 실행하지 않습니다.
  - YOLO 검출 결과는 완벽하지 않으므로 최종 판단은 사용자가 확인합니다.

예시 흐름:

```text
1. look_and_talk("앞에 뭐가 보여?")
2. 답변 확인
3. 안전하면 ask_navigation("앞으로 조금 가.")
4. 위험하면 ask_navigation("멈춰.")
```

In [ ]:
# 1. 카메라로 현재 상황 확인
answer, summary, annotated = look_and_talk("앞에 뭐가 보여?", show=True)
print(answer)

# 2. 안전하다고 판단되면 짧게 전진
answer = ask_navigation("앞으로 조금 가.")
print(answer)

# 3. 위험하다고 판단되면 정지
answer = ask_navigation("멈춰.")
print(answer)